### 퓨샷 프롬프트

In [1]:
from dotenv import load_dotenv
from langchain_teddynote import logging
from langchain_openai import ChatOpenAI
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_teddynote.messages import stream_response
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.prompts.few_shot import FewShotPromptTemplate

llm = ChatOpenAI()

logging.langsmith("test0914")

LangSmith 추적을 시작합니다.
[프로젝트명]
test0914


In [2]:
examples = [
    {
        "question": "스티브 잡스와 아인슈타인 중 누가 더 오래 살았나요 ?",
        "answer": """이 질문에 추가 질문이 필요한가요: 예.
                    추가 질문 : 스티브 잡스는 몇 살에 사망했나요 ?
                    중간 답변 : 스티브 잡스는 56세에 사망했습니다.
                    추가 질문 : 아인슈타인은 몇 살에 사망했나요
                    중간 답변 : 아인슈타인은 76세에 사망했습니다.
                    최종 답변은 : 아인슈타인"""
    }
]

In [3]:
example_prompt = PromptTemplate.from_template(
    "Question:\n{question}\nAnswer:\n{answer}"
)

print(example_prompt.format(**examples[0]))

Question:
스티브 잡스와 아인슈타인 중 누가 더 오래 살았나요 ?
Answer:
이 질문에 추가 질문이 필요한가요: 예.
                    추가 질문 : 스티브 잡스는 몇 살에 사망했나요 ?
                    중간 답변 : 스티브 잡스는 56세에 사망했습니다.
                    추가 질문 : 아인슈타인은 몇 살에 사망했나요
                    중간 답변 : 아인슈타인은 76세에 사망했습니다.
                    최종 답변은 : 아인슈타인


In [5]:
prompt = FewShotPromptTemplate(
    examples = examples,
    example_prompt = example_prompt,
    suffix = "Question:\n{question}\nAnswer:",
    input_variables = ["question"]
)

question = "Google이 창립된 연도에 Bill Gates의 나이는 몇 살인가요?"
final_prompt = prompt.format(question = question)

print(final_prompt)

Question:
스티브 잡스와 아인슈타인 중 누가 더 오래 살았나요 ?
Answer:
이 질문에 추가 질문이 필요한가요: 예.
                    추가 질문 : 스티브 잡스는 몇 살에 사망했나요 ?
                    중간 답변 : 스티브 잡스는 56세에 사망했습니다.
                    추가 질문 : 아인슈타인은 몇 살에 사망했나요
                    중간 답변 : 아인슈타인은 76세에 사망했습니다.
                    최종 답변은 : 아인슈타인

Question:
Google이 창립된 연도에 Bill Gates의 나이는 몇 살인가요?
Answer:


In [6]:
answer = llm.stream(final_prompt)
stream_response(answer)

이 질문에 추가 질문이 필요한가요: 예.
          추가 질문: Google이 창립된 연도는 언제인가요?
          중간 답변: Google이 1998년에 창립되었습니다.
          추가 질문: Bill Gates는 1998년에 몇 살이었나요?
          중간 답변: Bill Gates는 1998년에 42세였습니다.
          최종 답변은: 42세

In [8]:
prompt = FewShotPromptTemplate(
    examples = examples,
    example_prompt = example_prompt,
    suffix = "Question:\n{question}\nAnswer:",
    input_variables = ["question"]
)

chain = prompt | llm | StrOutputParser()

answer = chain.stream(
    {"question": "Google이 창립된 연도에 Bill Gates의 나이는 몇 살인가요 ?"}
)

stream_response(answer)

Google이 창립된 연도인 1998년에 Bill Gates는 42살이었습니다.